In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "data/sachs"

# ---------------------------------------------------------------------------
# Column order is fixed by the source files. iv=N.txt means "column index N
# (0-indexed in this list) is the intervention target". iv=.txt is observational.
# ---------------------------------------------------------------------------
node_names = ["Raf", "Mek", "PLCg", "PIP2", "PIP3", "Erk", "Akt", "PKA", "PKC", "p38", "JNK"]
p = len(node_names)  # 11

# (filename suffix, intervention target index, human-readable label)
ENV_SPEC = [
    ("",  None, "obs (cd3cd28 + cd3cd28icam2)"),
    ("1", 1,    "U0126 — Mek inhibitor"),
    ("3", 3,    "Psitectorigenin — PIP2 inhibitor"),
    ("4", 4,    "LY294002 — PIP3 inhibitor (via PI3K)"),
    ("6", 6,    "AKT inhibitor VIII — Akt inhibitor"),
    ("8", 8,    "Gö6976 — PKC inhibitor"),
]

# ---------------------------------------------------------------------------
# Load per-environment samples
# ---------------------------------------------------------------------------
X_list, env_labels, target_idx = [], [], []
for suffix, target, label in ENV_SPEC:
    df = pd.read_csv(os.path.join(DATA_DIR, f"iv={suffix}.txt"))
    assert list(df.columns) == node_names, f"column mismatch in iv={suffix}.txt"
    X_list.append(df.values.astype(float))
    env_labels.append(label)
    target_idx.append(target)

sample_sizes = [X.shape[0] for X in X_list]
n_envs = len(X_list)
print(f"{n_envs} environments, {p} nodes, sample sizes = {sample_sizes}, total = {sum(sample_sizes)}")

# ---------------------------------------------------------------------------
# Intervention-pattern matrix B, matching the LSEM convention in generate_LSEM.py:
#   B has shape (p, n_envs). B[i, j] = 1 means node i is NOT intervened in env j;
#   B[i, j] = 0 means node i IS intervened in env j. First column is observational.
# ---------------------------------------------------------------------------
B = np.ones((p, n_envs), dtype=int)
for j, t in enumerate(target_idx):
    if t is not None:
        B[t, j] = 0

print("\nIntervention pattern B (rows=nodes, cols=envs; 0 = intervened):")
print(pd.DataFrame(B, index=node_names, columns=[f"env{j}" for j in range(n_envs)]))

# Optional flat/long-form DataFrame with env + target columns, useful for plotting
flat = pd.concat(
    [pd.DataFrame(X, columns=node_names).assign(env=j, target=target_idx[j])
     for j, X in enumerate(X_list)],
    ignore_index=True,
)
print("\nflat shape:", flat.shape)
flat.head()


6 environments, 11 nodes, sample sizes = [1755, 799, 810, 848, 911, 723], total = 5846

Intervention pattern B (rows=nodes, cols=envs; 0 = intervened):
      env0  env1  env2  env3  env4  env5
Raf      1     1     1     1     1     1
Mek      1     0     1     1     1     1
PLCg     1     1     1     1     1     1
PIP2     1     1     0     1     1     1
PIP3     1     1     1     0     1     1
Erk      1     1     1     1     1     1
Akt      1     1     1     1     0     1
PKA      1     1     1     1     1     1
PKC      1     1     1     1     1     0
p38      1     1     1     1     1     1
JNK      1     1     1     1     1     1

flat shape: (5846, 13)


,Raf,Mek,PLCg,PIP2,PIP3,Erk,Akt,PKA,PKC,p38,JNK,env,target
0,3.273364,2.580217,2.177022,2.906901,4.074142,1.888584,2.833213,6.025866,2.833213,3.804438,3.688879,0,None
1,3.580737,2.803360,2.509599,2.821379,2.095561,2.923162,3.481240,5.863631,1.214913,2.803360,4.119037,0,None
2,4.084294,3.786460,2.681022,2.322388,2.564949,2.701361,3.481240,5.998937,2.433613,3.462606,2.970414,0,None
3,4.290459,4.416428,3.139833,2.602690,0.254642,1.763017,2.468100,6.269096,2.617396,3.353407,3.139833,0,None
4,3.517498,2.985682,1.646734,2.275214,3.210844,3.049273,3.830813,5.720312,1.539015,3.246491,4.398146,0,None


In [11]:
# unidentifiable groups 

from collections import defaultdict

groups = defaultdict(list)
for i, name in enumerate(node_names):
    pattern = tuple(B[i].tolist())
    groups[pattern].append(name)

print(f"{len(groups)} distinct intervention patterns across {p} nodes\n")
print(f"{'pattern (env0..env%d)' % (n_envs-1):<24}  size  members" )
print("-" * 70)
for pattern, members in sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    pat_str = "".join(str(b) for b in pattern)
    intervened_envs = [j for j, b in enumerate(pattern) if b == 0]
    iv_label = f"intervened in env{intervened_envs}" if intervened_envs else "never intervened (obs-only)"
    print(f"{pat_str:<24}  {len(members):<4}  {', '.join(members):<28}  {iv_label}")


6 distinct intervention patterns across 11 nodes

pattern (env0..env5)      size  members
----------------------------------------------------------------------
111111                    6     Raf, PLCg, Erk, PKA, p38, JNK  never intervened (obs-only)
101111                    1     Mek                           intervened in env[1]
110111                    1     PIP2                          intervened in env[2]
111011                    1     PIP3                          intervened in env[3]
111101                    1     Akt                           intervened in env[4]
111110                    1     PKC                           intervened in env[5]


In [9]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "scr")))


from TSCD import TSCD


# ---------------------------------------------------------------------------
# Run the method
# ---------------------------------------------------------------------------

Lambda_est, causal_order = TSCD(X_list, B, n_candidates=2, epsilon=0.1, alpha=0.05)

print("Estimated causal order (root → sink):")
print([node_names[i] for i in causal_order])

# ---------------------------------------------------------------------------
# Threshold and align conventions:
#   Lambda_est uses (row=child, column=parent), so transpose for parent→child.
# ---------------------------------------------------------------------------
THRESHOLD = 0.2   # tweak if needed
A_est = (np.abs(Lambda_est) > THRESHOLD).astype(int)   # now A_est[parent, child]


Estimated causal order (root → sink):
['PLCg', 'Akt', 'Mek', 'PIP3', 'p38', 'JNK', 'PKA', 'Erk', 'Raf', 'PIP2', 'PKC']


In [10]:
child, parent = np.where(A_est!=0)
for i in range(len(child)):
    print(f"{node_names[parent[i]]} -> {node_names[child[i]]} with weight {Lambda_est[child[i], parent[i]]:.3f}")

Mek -> Raf with weight 0.862
PLCg -> PIP2 with weight 0.280
PIP3 -> PIP2 with weight 0.428
PLCg -> PIP3 with weight 0.298
Akt -> Erk with weight 0.812
Akt -> PKA with weight 0.578
p38 -> PKC with weight 0.891
p38 -> JNK with weight 0.456
